<a href="https://colab.research.google.com/github/tajo2024231/The-Left-Hand-Rule/blob/main/%E5%B7%A6%E6%89%8B%E3%81%AE%E6%B3%95%E5%89%87.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# スペースキーで一時停止できます。

In [ ]:
import random
import tkinter as tk

class Application(tk.Frame):
    def __init__(self, master):
        super().__init__(master)
        self.master.geometry("800x600")
        self.create_widget()
        self.master.bind("<KeyPress>", self.key_event)

    def key_event(self, event):
        if event.keysym == "space" :
            if self.flag_move : self.flag_move = False
            else :
                self.flag_move = True
                self.puzzle()

    def create_widget(self) :
        self.canvas = tk.Canvas(self.master, width="800", height="600", bg="#ffffff")
        self.canvas.place(x=0, y=0)
        self.prepare_map()

    def prepare_map(self) :
        self.flag_move = False
        self.canvas.create_rectangle(130,30,670,570)
        self.map = [["#"]*9 for _ in range(9)] #奇数
        self.map[7][1] = " "
        self.B = [[[0,0],[1,2]],[[0,0],[-1,-2]],[[1,2],[0,0]],[[-1,-2],[0,0]]]
        self.make_map(7,1,self.B)
        if self.map[1][7] == " " :
            self.create_map()
            self.prepare_puzzle()
        else : self.prepare_map()

    def make_map(self, nx, ny, B) : #穴掘り法で迷路を作る
        random.shuffle(B)
        for i in range(4) :
            if nx + self.B[i][0][1] < 1 or nx + self.B[i][0][1] > 8 :
                continue
            if ny + self.B[i][1][1] < 1 or ny + self.B[i][1][1] > 8 :
                continue
            if self.map[nx + B[i][0][1]][ny + self.B[i][1][1]] == " " :
                continue
            self.map[nx + self.B[i][0][0]][ny + self.B[i][1][0]] = " "
            self.map[nx + self.B[i][0][1]][ny + self.B[i][1][1]] = " "
            self.make_map(nx + B[i][0][1], ny + B[i][1][1], self.B)

    def create_map(self) :
        for i in range(9) :
            for j in range(9) :
                if self.map[i][j] == "#" : self.canvas.create_rectangle(j*60+130,i*60+30,j*60+190,i*60+90,fill="#a0a0a0", tag="widget")
        self.canvas.create_rectangle(560,100,565,150,fill="#ffffff", tag="widget")
        self.canvas.create_rectangle(560,100,600,125,fill="#ff0000", tag="widget")

    def prepare_puzzle(self) :
        self.flag = True
        self.ny, self.nx = 7, 1 #現在地
        self.num_anpanman = 0 #アンパンマンの顔の向き 北、東、南、西
        self.X = [[0,1,-1], #北 上、右、左
                  [1,0,0], #東 上、右、左
                  [0,-1,1], #南 上、右、左
                  [-1,0,0]] #西 上、右、左

        self.Y = [[-1,0,0], #北 上、右、左
                  [0,1,-1], #東 上、右、左
                  [1,0,0], #南 上、右、左
                  [0,-1,1]] #西 上、右、左
        self.func = [self.create_anpanman_1, self.create_anpanman_2, self.create_anpanman_3, self.create_anpanman_4]
        self.create_anpanman_1()
        self.canvas.moveto("anpanman", self.nx*60+140, self.ny*60+35)
        self.flag_move = True
        self.master.after(1000,self.puzzle)





    def puzzle(self) :
        if self.map[self.ny + self.Y[self.num_anpanman][2]][self.nx + self.X[self.num_anpanman][2]] == " " : #左回転前進
            self.ny += self.Y[self.num_anpanman][2]
            self.nx += self.X[self.num_anpanman][2]
            self.num_anpanman += 3
            self.num_anpanman %= 4
            self.canvas.delete("anpanman")
            self.func[self.num_anpanman]() #作り直し
            self.canvas.moveto("anpanman", self.nx*60+140, self.ny*60+35)

        elif self.map[self.ny + self.Y[self.num_anpanman][0]][self.nx + self.X[self.num_anpanman][0]] == " " : #前進
            self.ny += self.Y[self.num_anpanman][0]
            self.nx += self.X[self.num_anpanman][0]
            self.canvas.move("anpanman", self.X[self.num_anpanman][0]*60, self.Y[self.num_anpanman][0]*60) #移動

        else : #右回転
            self.num_anpanman += 1
            self.num_anpanman %= 4
            self.canvas.delete("anpanman")
            self.func[self.num_anpanman]()  #作り直し
            self.canvas.moveto("anpanman", self.nx*60+140, self.ny*60+35)
        if self.num_anpanman == 0 and self.map[self.ny][self.nx-1] == "#" : self.canvas.create_line(self.nx*60+130, self.ny*60+30, self.nx*60+130, self.ny*60+90, width=5, fill="#ff0000", tag="widget")
        if self.num_anpanman == 1 and self.map[self.ny-1][self.nx] == "#" : self.canvas.create_line(self.nx*60+130, self.ny*60+30, self.nx*60+190, self.ny*60+30, width=5, fill="#ff0000", tag="widget")
        if self.num_anpanman == 2 and self.map[self.ny][self.nx+1] == "#" : self.canvas.create_line(self.nx*60+190, self.ny*60+30, self.nx*60+190, self.ny*60+90, width=5, fill="#ff0000", tag="widget")
        if self.num_anpanman == 3 and self.map[self.ny+1][self.nx] == "#" : self.canvas.create_line(self.nx*60+130, self.ny*60+90, self.nx*60+190, self.ny*60+90, width=5, fill="#ff0000", tag="widget")
        if self.ny == 1 and self.nx == 7 : self.flag = False
        if self.flag and self.flag_move : self.master.after(1000, self.puzzle)
        elif not self.flag : self.master.after(3000, self.ending)

    def ending(self) :
        self.canvas.delete("anpanman")
        self.canvas.delete("widget")
        self.prepare_map()



    def create_anpanman_1(self) :
        self.canvas.create_oval(0,0,40,40,fill="#deb068", tag="anpanman")
        self.canvas.create_oval(5,42,35,50,fill="#202020", tag="anpanman")
        self.canvas.create_oval(0,0,40,40,fill="#deb068", tag="anpanman")

    def create_anpanman_2(self) :
        self.canvas.create_oval(0,0,40,40,fill="#deb068", tag="anpanman")
        self.canvas.create_oval(5,42,35,50,fill="#202020", tag="anpanman")
        self.canvas.create_oval(0,0,40,40,fill="#deb068", tag="anpanman")
        self.canvas.create_oval(33,10,35,15,fill="#000000", tag="anpanman")
        self.canvas.create_oval(26,17,36,27,fill="#ff4500", tag="anpanman")
        self.canvas.create_oval(35,17,45,27,fill="#ff0000", tag="anpanman")
        self.canvas.create_arc(26,24,36,34,fill="#c71585", start=0,extent=-90, tag="anpanman")

    def create_anpanman_3(self) :
        self.canvas.create_oval(0,0,40,40,fill="#deb068", tag="anpanman")
        self.canvas.create_oval(5,42,35,50,fill="#202020", tag="anpanman")
        self.canvas.create_oval(0,0,40,40,fill="#deb068", tag="anpanman")
        self.canvas.create_oval(14,10,16,15,fill="#000000", tag="anpanman")
        self.canvas.create_oval(24,10,26,15,fill="#000000", tag="anpanman")
        self.canvas.create_oval(5,17,15,27,fill="#ff4500", tag="anpanman")
        self.canvas.create_oval(25,17,35,27,fill="#ff4500", tag="anpanman")
        self.canvas.create_oval(14,17,26,27,fill="#ff0000", tag="anpanman")
        self.canvas.create_arc(15,25,25,35,fill="#c71585", extent=-180, tag="anpanman")

    def create_anpanman_4(self) :
        self.canvas.create_oval(0,0,40,40,fill="#deb068", tag="anpanman")
        self.canvas.create_oval(5,42,35,50,fill="#202020", tag="anpanman")
        self.canvas.create_oval(0,0,40,40,fill="#deb068", tag="anpanman")
        self.canvas.create_oval(5,10,7,15,fill="#000000", tag="anpanman")
        self.canvas.create_oval(4,17,14,27,fill="#ff4500", tag="anpanman")
        self.canvas.create_oval(-5,17,5,27,fill="#ff0000", tag="anpanman")
        self.canvas.create_arc(4,24,14,34,fill="#c71585", start=180,extent=90, tag="anpanman")

if __name__ == "__main__":
    root = tk.Tk()
    myapp = Application(master = root)
    myapp.mainloop()